# 4.1 검색/연구 에이전트

검색 에이전트는 핵심 딜레마를 드러낸다:
- **외부 지식 접근**: 포괄적이지만 훈련 불안정성, API 비용, 신뢰도 검증의 어려움
- **내부 지식(메모리)**: 통제되지만 환각 위험, 확장성 제약

이 모듈에서는 **Perceive → Memory → Retrieve → Plan → Act** 사이클을 구현하고, **다단계 추론의 신용 할당 문제**를 다룬다.

**핵심 아이디어**: 검색의 각 홉(Hop)이 최종 보상에 얼마나 기여했는가?

### 외부 vs 내부 지식의 트레이드오프

```
외부 지식 (Web Search):
  장점: 최신 정보, 포괄적
  단점: 비용(API 호출), 신뢰도 검증, 훈련 불안정

내부 지식 (메모리):
  장점: 빠름, 안정적
  단점: 제한적, 환각 위험
```

### 다단계 추론과 신용 할당

```
Query → [Hop1] → [Hop2] → [Hop3] → Answer
                                      ↓
                              Final Reward R

신용 할당 문제:
  어느 홉이 R에 가장 기여했는가?
  Credit(Hop_i) = ?
```

In [1]:
from utils_openai import *
import time
from typing import List, Dict

heading("준비: 검색 에이전트 라이브러리")
print("✓ ask() - LLM 호출")
print("✓ chat() - 멀티턴 대화")
print("✓ MemoryStream - 외부 지식과 내부 기억 혼합")
print("✓ 다단계 추론 추적 가능")


────────────────────────────────────────
  준비: 검색 에이전트 라이브러리
────────────────────────────────────────
✓ ask() - LLM 호출
✓ chat() - 멀티턴 대화
✓ MemoryStream - 외부 지식과 내부 기억 혼합
✓ 다단계 추론 추적 가능


## 멀티홉 검색과 신용 할당

### 공식: 신용 할당

```
각 홉의 기여도:
  relevance_score[i] = (출력이 다음 홉의 입력으로 얼마나 잘 사용되었는가)
  credit[i] = relevance_score[i] / Σ(relevance_scores)

최종 보상:
  reward = Σ(credit[i] × LLM_eval(result, criteria))
```

### 검증 패러독스

- 정보가 정확한지 확인하려면 **다른 소스와 교차확인** 필요 → 추가 홉
- 하지만 교차확인 자체가 실패할 수 있음 → 악순환

## 실습 1: Perceive → Memory → Retrieve 사이클

기본 메모리 기반 검색을 구현한다.

In [3]:
heading("실습 1: 메모리 기반 검색")

# 초기 메모리 구성 (도메인별)
search_memory = MemoryStream()

# 사전 저장된 신뢰할 수 있는 정보들
knowledge_base = [
    ("LLM은 트랜스포머 아키텍처 기반이다.", 0.95, "foundational"),
    ("GPT-4는 OpenAI의 최신 모델이다.", 0.90, "current"),
    ("강화학습은 보상 신호로 정책을 최적화한다.", 0.92, "foundational"),
    ("에이전트는 다단계 추론이 가능하다.", 0.85, "emerging"),
]

step_print(1, "메모리 초기화", f"{len(knowledge_base)}개 기초 지식 저장")

for content, importance, mem_type in knowledge_base:
    search_memory.add(content, importance=importance, mem_type=mem_type)
    print(f"  ✓ [{mem_type:12s}] {content}")

print(f"메모리 크기: {len(search_memory)}")


────────────────────────────────────────
  실습 1: 메모리 기반 검색
────────────────────────────────────────
  [Step 1] 메모리 초기화
    4개 기초 지식 저장
  ✓ [foundational] LLM은 트랜스포머 아키텍처 기반이다.
  ✓ [current     ] GPT-4는 OpenAI의 최신 모델이다.
  ✓ [foundational] 강화학습은 보상 신호로 정책을 최적화한다.
  ✓ [emerging    ] 에이전트는 다단계 추론이 가능하다.
메모리 크기: 4


In [5]:
# 사용자 쿼리를 처리하는 Perceive → Retrieve 사이클
step_print(2, "쿼리 처리", "Perceive → Retrieve 사이클")

queries = [
    "LLM의 기본 구조는 무엇인가?",
    "강화학습과 에이전트의 관계는?",
    "최신 AI 발전은?"
]

retrieved_results = {}
for q in queries:
    # Perceive: 쿼리 인식
    search_memory.add(f"[쿼리] {q}", importance=0.8, mem_type="query")
    
    # Retrieve: 관련 메모리 검색
    results = search_memory.retrieve(q, top_k=2)
    retrieved_results[q] = results
    
    print(f"  질문: {q}")
    print(f"  검색 결과 ({len(results)}개):")
    for r in results:
        print(f"    - {r['content'][:60]}... (중요도 {r['importance']})")

  [Step 2] 쿼리 처리
    Perceive → Retrieve 사이클
  질문: LLM의 기본 구조는 무엇인가?
  검색 결과 (2개):
    - [쿼리] LLM의 기본 구조는 무엇인가?... (중요도 0.8)
    - LLM은 트랜스포머 아키텍처 기반이다.... (중요도 0.95)
  질문: 강화학습과 에이전트의 관계는?
  검색 결과 (2개):
    - [쿼리] 강화학습과 에이전트의 관계는?... (중요도 0.8)
    - [쿼리] LLM의 기본 구조는 무엇인가?... (중요도 0.8)
  질문: 최신 AI 발전은?
  검색 결과 (2개):
    - [쿼리] 최신 AI 발전은?... (중요도 0.8)
    - GPT-4는 OpenAI의 최신 모델이다.... (중요도 0.9)


## 실습 2: Plan → Act (LLM 기반 추론)

LLM이 검색 결과를 바탕으로 답변을 생성하고 추가 질문을 계획한다.

In [8]:
heading("실습 2: LLM 기반 계획과 실행")

def plan_and_answer(query: str, retrieved_context: List[Dict]) -> Dict:
    """검색 결과를 바탕으로 답변을 생성한다."""
    context_text = "".join([f"- {m['content']}" for m in retrieved_context])
    
    prompt = f"""
당신은 검색 에이전트의 추론 모듈이다.

질문: {query}

이용 가능한 정보:
{context_text}

이 정보를 바탕으로 1~2문장으로 답변하고, 추가 확인이 필요한 부분을 언급하다.
각 문장을 '~다'로 끝내다.
"""
    
    answer = ask(prompt, temperature=0.3)
    return {"query": query, "answer": answer, "context_count": len(retrieved_context)}

step_print(1, "답변 생성", "LLM이 검색 결과를 바탕으로 추론")

answers = []
for q in queries:
    context = retrieved_results.get(q, [])
    result = plan_and_answer(q, context)
    answers.append(result)
    print(f"[Q] {q}")
    print(f"  [A] {result['answer']}")


────────────────────────────────────────
  실습 2: LLM 기반 계획과 실행
────────────────────────────────────────
  [Step 1] 답변 생성
    LLM이 검색 결과를 바탕으로 추론
[Q] LLM의 기본 구조는 무엇인가?
  [A] LLM의 기본 구조는 트랜스포머 아키텍처 기반이다. 추가적으로, 트랜스포머의 세부 구성 요소나 작동 원리에 대한 정보가 필요하다.
[Q] 강화학습과 에이전트의 관계는?
  [A] 강화학습은 에이전트가 환경과 상호작용하며 보상을 최대화하는 행동을 학습하는 방법론이다. 추가적으로, 에이전트의 구조와 작동 방식에 대한 구체적인 정보가 필요하다.
[Q] 최신 AI 발전은?
  [A] 최신 AI 발전으로는 OpenAI의 GPT-4 모델이 있다. 추가적으로 다른 AI 기술이나 발전에 대한 정보도 확인할 필요가 있다.


## 실습 3: 다단계 추론과 신용 할당

여러 검색 홉을 거쳐 답변에 도달할 때, 각 홉의 기여도를 계산한다.

In [10]:
heading("실습 3: 멀티홉 신용 할당")

# 멀티홉 추론 시뮬레이션
def simulate_multihop_search(initial_query: str, num_hops: int = 3) -> Dict:
    """여러 단계의 검색을 시뮬레이션하고 신용을 할당한다."""
    hops = []
    
    step_print(1, f"{initial_query}의 {num_hops}홉 추론", "각 홉 평가")
    
    for hop_num in range(1, num_hops + 1):
        # 각 홉의 관련성과 품질 평가
        relevance = 1.0 - (hop_num - 1) * 0.15  # 나중 홉일수록 관련성 감소
        quality = 0.85 + (hop_num - 1) * 0.05   # 나중 홉일수록 품질 향상
        
        hops.append({
            "hop": hop_num,
            "relevance": relevance,
            "quality": quality,
            "contribution": relevance * quality
        })
    
    # 신용 할당: 정규화
    total_contribution = sum(h["contribution"] for h in hops)
    for hop in hops:
        hop["credit"] = hop["contribution"] / total_contribution
    
    return {"query": initial_query, "hops": hops, "total_credit": 1.0}

# 멀티홉 실행
multihop_results = simulate_multihop_search("LLM 에이전트의 강화학습", num_hops=3)

for hop in multihop_results["hops"]:
    print(f"    Hop {hop['hop']}:")
    print(f"    관련성: {hop['relevance']:.2f}")
    print(f"    품질: {hop['quality']:.2f}")
    print(f"    신용: {hop['credit']:.2%}")


────────────────────────────────────────
  실습 3: 멀티홉 신용 할당
────────────────────────────────────────
  [Step 1] LLM 에이전트의 강화학습의 3홉 추론
    각 홉 평가
    Hop 1:
    관련성: 1.00
    품질: 0.85
    신용: 37.28%
    Hop 2:
    관련성: 0.85
    품질: 0.90
    신용: 33.55%
    Hop 3:
    관련성: 0.70
    품질: 0.95
    신용: 29.17%


In [13]:
step_print(2, "신용 할당 분석", "각 홉의 기여도 시각화")

# 신용 할당 요약
print(f" 신용 할당 분포:")
for i, hop in enumerate(multihop_results["hops"], 1):
    bar = "█" * int(hop["credit"] * 30)
    print(f"    Hop {i}: {bar} {hop['credit']:.1%}")

print(f"    → 초반 홉이 더 높은 신용을 받음")
print(f"    이것이 '다단계 추론에서 초기 검색 단계가 중요'한 이유")

  [Step 2] 신용 할당 분석
    각 홉의 기여도 시각화
 신용 할당 분포:
    Hop 1: ███████████ 37.3%
    Hop 2: ██████████ 33.6%
    Hop 3: ████████ 29.2%
    → 초반 홉이 더 높은 신용을 받음
    이것이 '다단계 추론에서 초기 검색 단계가 중요'한 이유


## 실습 4: 검증 패러독스와 신뢰도

정보의 정확성을 검증할수록 비용(추가 홉)이 증가하는 딜레마를 다룬다.

In [16]:
heading("실습 4: 검증 패러독스")

def analyze_verification_paradox() -> Dict:
    """정보 검증 비용과 신뢰도의 트레이드오프를 분석한다."""
    
    verification_levels = [
        {"level": "신뢰기본값", "sources": 1, "cost": 1, "confidence": 0.70},
        {"level": "1차검증", "sources": 2, "cost": 2, "confidence": 0.82},
        {"level": "2차검증", "sources": 3, "cost": 3, "confidence": 0.90},
        {"level": "3차검증", "sources": 4, "cost": 4, "confidence": 0.93},
    ]
    
    step_print(1, "검증 비용 분석", "각 검증 단계별 비용과 신뢰도")
    
    for v in verification_levels:
        marginal_gain = v["confidence"] - (verification_levels[0]["confidence"] if v["level"] != "신뢰기본값" else 0)
        efficiency = (v["confidence"] - verification_levels[0]["confidence"]) / (v["cost"] - 1) if v["cost"] > 1 else 0
        
        print(f"    [{v['level']}]")
        print(f"    소스 수: {v['sources']}개")
        print(f"    API 비용: {v['cost']}배")
        print(f"    신뢰도: {v['confidence']:.0%}")
        if v["cost"] > 1:
            print(f"    효율성: {efficiency:.3f} (신뢰도/비용)")
    
    return verification_levels

verification_data = analyze_verification_paradox()

print(f"    → 신뢰도가 점점 더 천천히 증가한다 (수확체감)")
print(f"    이것이 '검증 패러독스': 완벽한 검증은 불가능하고 비용이 높다")


────────────────────────────────────────
  실습 4: 검증 패러독스
────────────────────────────────────────
  [Step 1] 검증 비용 분석
    각 검증 단계별 비용과 신뢰도
    [신뢰기본값]
    소스 수: 1개
    API 비용: 1배
    신뢰도: 70%
    [1차검증]
    소스 수: 2개
    API 비용: 2배
    신뢰도: 82%
    효율성: 0.120 (신뢰도/비용)
    [2차검증]
    소스 수: 3개
    API 비용: 3배
    신뢰도: 90%
    효율성: 0.100 (신뢰도/비용)
    [3차검증]
    소스 수: 4개
    API 비용: 4배
    신뢰도: 93%
    효율성: 0.077 (신뢰도/비용)
    → 신뢰도가 점점 더 천천히 증가한다 (수확체감)
    이것이 '검증 패러독스': 완벽한 검증은 불가능하고 비용이 높다


## 요약

1. **Perceive → Memory → Retrieve → Plan → Act 사이클**
   - 각 단계가 에이전트의 의사결정을 형성한다.
   - 메모리 크기와 검색 품질이 성능을 결정한다.

2. **다단계 추론의 신용 할당**
   - 초반 홉이 더 높은 신용을 받는 경향이 있다.
   - 신용(hop_i) = 관련성 × 품질 / Σ기여도

3. **검증 패러독스**
   - 정보 검증은 필요하지만 비용이 가파르게 증가한다.
   - 완벽한 검증은 불가능 → 적절한 수준의 검증 기준 필요

4. **외부 vs 내부 지식의 균형**
   - 메모리(내부): 빠르고 안정적이지만 제한적
   - 검색(외부): 포괄적이지만 비용과 신뢰도 문제